# SQLite Database Operations

## Objective
This notebook covers database connection, table creation, records insertion, querying, modification, and Pandas integration.


---

## 1. Database Connection & Setup

This section connects to a SQLite database and creates the schema.

In [1]:
import sqlite3
import os

DB_PATH = '../artifacts/tmdb_movies.db'
os.makedirs('../artifacts', exist_ok=True)

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS movies")
cursor.execute("""
CREATE TABLE IF NOT EXISTS movies (
    id          INTEGER PRIMARY KEY AUTOINCREMENT,
    title       TEXT    NOT NULL,
    genre       TEXT,
    year        INTEGER,
    rating      REAL,
    popularity  REAL,
    revenue     INTEGER
)
""")
conn.commit()
print("Table 'movies' created successfully.")

Table 'movies' created successfully.


---

## 2. Inserting Records

This section populates the table with sample movie records.

In [2]:
sample_records = [
    ('Inception',         'Sci-Fi|Thriller', 2010, 8.8, 29.10,  836836967),
    ('The Dark Knight',   'Action|Drama',    2008, 9.0, 25.50, 1004934033),
    ('Interstellar',      'Sci-Fi|Drama',    2014, 8.6, 32.00,  701729206),
    ('Avatar',            'Action|Sci-Fi',   2009, 7.9, 185.01,2923706026),
    ('Titanic',           'Romance|Drama',   1997, 7.8, 41.47, 2264743305),
    ('Parasite',          'Thriller|Drama',  2019, 8.5, 52.30,  263000000),
    ('Avengers: Endgame', 'Action|Sci-Fi',   2019, 8.4, 163.02,2797800564),
    ('Joker',             'Thriller|Drama',  2019, 8.4, 60.00, 1074458282),
    ('Dune',              'Sci-Fi|Action',   2021, 8.0, 89.40,  401764958),
    ('Oppenheimer',       'Drama|History',   2023, 8.5, 110.50, 952000000),
]

cursor.executemany(
    "INSERT INTO movies (title, genre, year, rating, popularity, revenue) VALUES (?,?,?,?,?,?)",
    sample_records
)
conn.commit()
print(f"Inserted {len(sample_records)} records.")

Inserted 10 records.


---

## 3. Querying & Aggregations

This section retrieves and aggregates database records.

In [3]:
print("── SELECT ALL ──")
cursor.execute("SELECT id, title, year, rating FROM movies ORDER BY rating DESC")
rows = cursor.fetchall()
print(f"  {'ID':<4} {'Title':<25} {'Year':>6} {'Rating':>8}")
print(f"  {'-'*4} {'-'*25} {'-'*6} {'-'*8}")
for row in rows:
    print(f"  {row[0]:<4} {row[1]:<25} {row[2]:>6} {row[3]:>8.1f}")

print("\n── SELECT with WHERE ──")
cursor.execute("SELECT title, rating FROM movies WHERE rating >= 8.5 ORDER BY rating DESC")
top = cursor.fetchall()
print(f"  Movies with rating ≥ 8.5: {[r[0] for r in top]}")

print("\n── AGGREGATE FUNCTIONS ──")
cursor.execute("SELECT COUNT(*), AVG(rating), MAX(rating), MIN(rating) FROM movies")
agg = cursor.fetchone()
print(f"  Count: {agg[0]} | Avg: {agg[1]:.2f} | Max: {agg[2]} | Min: {agg[3]}")

print("\n── GROUP BY ──")
cursor.execute("""
    SELECT genre, COUNT(*) as cnt, ROUND(AVG(rating), 2) as avg_r
    FROM movies GROUP BY genre ORDER BY avg_r DESC
""")
for row in cursor.fetchall():
    print(f"  {row[0]:<25} count={row[1]}  avg_rating={row[2]}")

── SELECT ALL ──
  ID   Title                       Year   Rating
  ---- ------------------------- ------ --------
  2    The Dark Knight             2008      9.0
  1    Inception                   2010      8.8
  3    Interstellar                2014      8.6
  6    Parasite                    2019      8.5
  10   Oppenheimer                 2023      8.5
  7    Avengers: Endgame           2019      8.4
  8    Joker                       2019      8.4
  9    Dune                        2021      8.0
  4    Avatar                      2009      7.9
  5    Titanic                     1997      7.8

── SELECT with WHERE ──
  Movies with rating ≥ 8.5: ['The Dark Knight', 'Inception', 'Interstellar', 'Parasite', 'Oppenheimer']

── AGGREGATE FUNCTIONS ──
  Count: 10 | Avg: 8.39 | Max: 9.0 | Min: 7.8

── GROUP BY ──
  Action|Drama              count=1  avg_rating=9.0
  Sci-Fi|Thriller           count=1  avg_rating=8.8
  Sci-Fi|Drama              count=1  avg_rating=8.6
  Drama|History      

---

## 4. Modifying Data

This section covers updates and deletions.

In [4]:
cursor.execute("UPDATE movies SET rating = 9.1 WHERE title = 'The Dark Knight'")
conn.commit()
cursor.execute("SELECT title, rating FROM movies WHERE title='The Dark Knight'")
print(f"UPDATE → The Dark Knight new rating: {cursor.fetchone()[1]}")

cursor.execute("DELETE FROM movies WHERE rating < 7.9")
conn.commit()
cursor.execute("SELECT COUNT(*) FROM movies")
print(f"DELETE → Movies remaining (rating ≥ 7.9): {cursor.fetchone()[0]}")

UPDATE → The Dark Knight new rating: 9.1
DELETE → Movies remaining (rating ≥ 7.9): 9


---

## 5. Integrating with Pandas

This section loads query results into a Pandas DataFrame and closes the database connection.

In [5]:
import pandas as pd

db_df = pd.read_sql_query("SELECT * FROM movies ORDER BY rating DESC", conn)
print("── Final DB Table ──")
print(db_df.to_string(index=False))

conn.close()
print("\nSQLite connection closed.")

── Final DB Table ──
 id             title           genre  year  rating  popularity    revenue
  2   The Dark Knight    Action|Drama  2008     9.1       25.50 1004934033
  1         Inception Sci-Fi|Thriller  2010     8.8       29.10  836836967
  3      Interstellar    Sci-Fi|Drama  2014     8.6       32.00  701729206
  6          Parasite  Thriller|Drama  2019     8.5       52.30  263000000
 10       Oppenheimer   Drama|History  2023     8.5      110.50  952000000
  7 Avengers: Endgame   Action|Sci-Fi  2019     8.4      163.02 2797800564
  8             Joker  Thriller|Drama  2019     8.4       60.00 1074458282
  9              Dune   Sci-Fi|Action  2021     8.0       89.40  401764958
  4            Avatar   Action|Sci-Fi  2009     7.9      185.01 2923706026

SQLite connection closed.
